Mount Google Drive

In [1]:
import os
import re
import unicodedata

import pandas as pd

from google.colab import drive

drive.mount('/content/drive')

ROOT = "/content/drive/MyDrive/fever_project"

os.makedirs(f"{ROOT}/data/processed", exist_ok=True)

print("Project directory ready.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project directory ready.


Load datasets

In [2]:
# Load original FEVER datasets

train_df = pd.read_parquet(f"{ROOT}/data/processed/claims_train_flat.parquet")

val_df   = pd.read_parquet(f"{ROOT}/data/processed/claims_val_flat.parquet")
# Load extracted Wikipedia sentences

sentences_train = pd.read_parquet(
    f"{ROOT}/data/evidence/sentences_train.parquet"
)

sentences_val = pd.read_parquet(
    f"{ROOT}/data/evidence/sentences_val.parquet"
)

print("TRAIN CLAIMS:", train_df.shape)
print("VALIDATION CLAIMS:", val_df.shape)

print("TRAIN SENTENCES:", sentences_train.shape)
print("VALIDATION SENTENCES:", sentences_val.shape)

TRAIN CLAIMS: (311431, 7)
VALIDATION CLAIMS: (18999, 7)
TRAIN SENTENCES: (34796, 3)
VALIDATION SENTENCES: (3493, 3)


Prepare the page-title normalization

In [3]:
BR = {
    "-LRB-": "(",
    "-RRB-": ")",
    "-LSB-": "[",
    "-RSB-": "]",
    "-LCB-": "{",
    "-RCB-": "}"
}

def nfc(s):
    return unicodedata.normalize("NFC", s)

def clean(t):
    for k, v in BR.items():
        t = t.replace(k, v)

    return re.sub(r"\s+", " ", t).strip()

Build the gold pairs

In [4]:
def build_gold(df, sent):

    # Exclude NEI claims because they do not have gold evidence
    d = df[
        df["label"] != "NOT ENOUGH INFO"
    ].copy()

    # Normalize Wikipedia page titles
    d["page"] = d["evidence_wiki_url"].map(nfc)

    # Remove repeated evidence pointers
    d = d.drop_duplicates(
        ["id", "page", "evidence_sentence_id"]
    )

    # Match evidence pointers with extracted sentences
    d = d.merge(
        sent,
        how="left",
        left_on=["page", "evidence_sentence_id"],
        right_on=["page", "sent_id"]
    )

    # Group evidence by claim
    g = d.groupby("id", sort=False)

    out = g.agg(
        claim=("claim", "first"),
        label=("label", "first"),
        n_sent=("text", "count"),
        n_missing=("text", lambda s: s.isna().sum())
    ).reset_index()

    # Combine evidence sentences into one passage
    out["evidence"] = out["id"].map(
        g["text"].apply(
            lambda s: "\n".join(s.dropna())
        )
    )

    # Drop claims if any evidence sentence is missing
    bad = (
        (out["n_missing"] > 0) |
        (out["evidence"].str.len() == 0)
    )

    print(
        "Dropped for missing evidence:",
        int(bad.sum()),
        "of",
        len(out)
    )

    out = out[~bad].copy()

    out["evidence_source"] = "gold"

    return out[
        [
            "id",
            "claim",
            "evidence",
            "label",
            "n_sent",
            "evidence_source"
        ]
    ]

Generate training and validation gold pairs

In [5]:
gold_train = build_gold(
    train_df,
    sentences_train
)

gold_val = build_gold(
    val_df,
    sentences_val
)

print("\nTRAIN LABEL COUNTS:")
print(gold_train["label"].value_counts())

print("\nVALIDATION LABEL COUNTS:")
print(gold_val["label"].value_counts())

Dropped for missing evidence: 0 of 109810
Dropped for missing evidence: 0 of 6666

TRAIN LABEL COUNTS:
label
SUPPORTS    80035
REFUTES     29775
Name: count, dtype: int64

VALIDATION LABEL COUNTS:
label
SUPPORTS    3333
REFUTES     3333
Name: count, dtype: int64


Inspect the number of evidence sentences

In [6]:
print("TRAIN EVIDENCE SENTENCE SUMMARY:")
print(gold_train["n_sent"].describe())

print("\nVALIDATION EVIDENCE SENTENCE SUMMARY:")
print(gold_val["n_sent"].describe())

TRAIN EVIDENCE SENTENCE SUMMARY:
count    109810.000000
mean          1.858128
std           2.409084
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max          48.000000
Name: n_sent, dtype: float64

VALIDATION EVIDENCE SENTENCE SUMMARY:
count    6666.000000
mean        1.674467
std         1.992695
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max        52.000000
Name: n_sent, dtype: float64


Inspect three sample gold pairs

In [7]:
print("SAMPLE GOLD TRAINING PAIRS:")

print(
    gold_train.sample(
        min(3, len(gold_train)),
        random_state=0
    ).to_string(index=False)
)

SAMPLE GOLD TRAINING PAIRS:
    id                                                          claim                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 evidence    label  n_sent evidence_source
179974              Kirk Douglas was conceived on December 9th, 1916.                                                                                                                                                                                                                                                                                                       

Check for remaining FEVER bracket tokens

In [8]:
tokens = ["-LRB-", "-RRB-", "-LSB-", "-RSB-", "-LCB-", "-RCB-", "-COLON-"]
for name, g in (("train", gold_train), ("val", gold_val)):
    for col in ("claim", "evidence"):
        print(name, col, {t: int(g[col].str.contains(t, regex=False).sum()) for t in tokens})

train claim {'-LRB-': 0, '-RRB-': 0, '-LSB-': 0, '-RSB-': 0, '-LCB-': 0, '-RCB-': 0, '-COLON-': 0}
train evidence {'-LRB-': 0, '-RRB-': 0, '-LSB-': 0, '-RSB-': 0, '-LCB-': 0, '-RCB-': 0, '-COLON-': 0}
val claim {'-LRB-': 0, '-RRB-': 0, '-LSB-': 0, '-RSB-': 0, '-LCB-': 0, '-RCB-': 0, '-COLON-': 0}
val evidence {'-LRB-': 0, '-RRB-': 0, '-LSB-': 0, '-RSB-': 0, '-LCB-': 0, '-RCB-': 0, '-COLON-': 0}


In [9]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def trunc_report(g, name, max_len=128):
    enc = tok(g["claim"].tolist(), g["evidence"].tolist(), add_special_tokens=True, truncation=False)
    lens = pd.Series([len(x) for x in enc["input_ids"]], index=g.index)
    over = lens > max_len
    print(f"[{name}] tokens (claim+evidence): mean={lens.mean():.1f}, median={lens.median():.0f}, "
          f"p95={lens.quantile(.95):.0f}, max={lens.max()}")
    print(f"  truncated at {max_len}: {over.sum():,} of {len(g):,} ({over.mean():.1%})")
    print("  by label:", over.groupby(g["label"]).mean().round(3).to_dict())
    print("  by n_sent bucket:", over.groupby(pd.cut(g["n_sent"], [0,1,2,3,100])).mean().round(3).to_dict())

trunc_report(gold_train, "train")
trunc_report(gold_val, "val")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (618 > 512). Running this sequence through the model will result in indexing errors


[train] tokens (claim+evidence): mean=78.8, median=56, p95=205, max=1742
  truncated at 128: 11,765 of 109,810 (10.7%)
  by label: {'REFUTES': 0.11, 'SUPPORTS': 0.106}
  by n_sent bucket: {Interval(0, 1, closed='right'): 0.005, Interval(1, 2, closed='right'): 0.064, Interval(2, 3, closed='right'): 0.299, Interval(3, 100, closed='right'): 0.891}


/tmp/ipykernel_7144/4121620553.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print("  by n_sent bucket:", over.groupby(pd.cut(g["n_sent"], [0,1,2,3,100])).mean().round(3).to_dict())


[val] tokens (claim+evidence): mean=67.9, median=50, p95=166, max=1659
  truncated at 128: 515 of 6,666 (7.7%)
  by label: {'REFUTES': 0.079, 'SUPPORTS': 0.076}
  by n_sent bucket: {Interval(0, 1, closed='right'): 0.002, Interval(1, 2, closed='right'): 0.038, Interval(2, 3, closed='right'): 0.245, Interval(3, 100, closed='right'): 0.844}


/tmp/ipykernel_7144/4121620553.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print("  by n_sent bucket:", over.groupby(pd.cut(g["n_sent"], [0,1,2,3,100])).mean().round(3).to_dict())


In [10]:
gold_train.to_parquet(
    f"{ROOT}/data/processed/gold_train.parquet",
    index=False
)

gold_val.to_parquet(
    f"{ROOT}/data/processed/gold_val.parquet",
    index=False
)

print("Gold training and validation datasets saved successfully.")

Gold training and validation datasets saved successfully.
